In [1]:
!pip install torch torchvision scikit-learn matplotlib scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 57.2 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.9.41
    Uninstalling nvidia-nvjitlink-cu12-12.9.41:
      Successfully uninstalled nvidia-nvjitlink-cu12-12.9.41
  Attempting uninstall: nvidia-curand-cu12
    Found existing installation: nvidia-curand-cu12 10.3.10.19
    Uninstalling nvidia-curand-cu12-10.3.10.19:
      Successfully uninstalled nvidia-curand-cu12-10.3.10.19
  Attempt

In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import time
import sys
import os
from torch import optim
from torch.nn import (
    Module, Sequential, Conv2d, ReLU, AdaptiveAvgPool2d, 
    AdaptiveMaxPool2d, CrossEntropyLoss, AvgPool2d, MaxPool2d, 
    Linear, Sigmoid, Softmax, Dropout, Parameter
)
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from sklearn import metrics, preprocessing
from scipy.io import savemat

# Append module path
sys.path.append('../global_module/')
import datetime
import matplotlib.pyplot as plt
from operator import truediv
import scipy.io as sio
import math
# from global_module.Utils import extract_samll_cubic
import torch.utils.data as Data
import torch.nn.functional as F
from sklearn.decomposition import PCA
from collections import Counter

# Custom modules (commented out; you can uncomment if files exist)
# from CSANet_contrastive import Finalmodel
# from CSANetMain import attontion
# from generate_pic_contrastive import aa_and_each_accuracy, sampling, load_dataset, generate_png, generate_iter


In [3]:
def index_assignment(index, row, col, pad_length):
    new_assign = {}
    for counter, value in enumerate(index):
        assign_0 = value // col + pad_length
        assign_1 = value % col + pad_length
        new_assign[counter] = [assign_0, assign_1]
    return new_assign


def assignment_index(assign_0, assign_1, col):
    new_index = assign_0 * col + assign_1
    return new_index


def select_patch(matrix, pos_row, pos_col, ex_len):
    selected_rows = matrix[range(pos_row-ex_len, pos_row+ex_len+1)]
    selected_patch = selected_rows[:, range(pos_col-ex_len, pos_col+ex_len+1)]
    return selected_patch


def select_small_cubic(data_size, data_indices, whole_data, patch_length, padded_data, dimension):
    small_cubic_data = np.zeros((data_size, 2 * patch_length + 1, 2 * patch_length + 1, dimension))
    data_assign = index_assignment(data_indices, whole_data.shape[0], whole_data.shape[1], patch_length)
    for i in range(len(data_assign)):
        small_cubic_data[i] = select_patch(padded_data, data_assign[i][0], data_assign[i][1], patch_length)
    return small_cubic_data



In [4]:
import scipy.io as sio
import math

def load_dataset():
    T1_ori = sio.loadmat('/kaggle/input/sst-change-detection-dataset/ChangeDetectionDataset-master-72b0263758e1f4dc04769bac5a688f951f5ce23a/Hermiston/hermiston2004.mat')
    T2_ori = sio.loadmat('/kaggle/input/sst-change-detection-dataset/ChangeDetectionDataset-master-72b0263758e1f4dc04769bac5a688f951f5ce23a/Hermiston/hermiston2007.mat')
    mat_gt = sio.loadmat('/kaggle/input/sst-change-detection-dataset/ChangeDetectionDataset-master-72b0263758e1f4dc04769bac5a688f951f5ce23a/Hermiston/rdChangesHermiston_5classes.mat')

    TT1 = T1_ori['T1']
    TT2 = T2_ori['T2']
    gt_hsi = mat_gt['GT']

    TOTAL_SIZE = TT1.shape[0] * TT1.shape[1]
    VALIDATION_SPLIT = 0.97
    TRAIN_SIZE = math.ceil(TOTAL_SIZE * VALIDATION_SPLIT)

    return TT1, TT2, gt_hsi, TOTAL_SIZE, TRAIN_SIZE, VALIDATION_SPLIT





def save_cmap(img, cmap, fname):
    sizes = np.shape(img)
    height = float(sizes[0])
    width = float(sizes[1])

    fig = plt.figure()
    fig.set_size_inches(width / height, 1, forward=False)
    ax = plt.Axes(fig, [0., 0., 1., 1.])
    ax.set_axis_off()
    fig.add_axes(ax)

    ax.imshow(img, cmap=cmap)
    plt.savefig(fname, dpi=height)
    plt.close()


def sampling(proportion, ground_truth):
    train = {}
    test = {}
    labels_loc = {}
    m = max(ground_truth)
    for i in range(m):
        indexes = [j for j, x in enumerate(ground_truth.ravel().tolist()) if x == i + 1]
        np.random.shuffle(indexes)
        labels_loc[i] = indexes
        if proportion != 1:
            nb_val = max(int((1 - proportion) * len(indexes)), 3)
        else:
            nb_val = 0
        # print(i, nb_val, indexes[:nb_val])
        # train[i] = indexes[:-nb_val]
        # test[i] = indexes[-nb_val:]
        train[i] = indexes[:nb_val]
        test[i] = indexes[nb_val:]
    train_indexes = []
    test_indexes = []
    for i in range(m):
        train_indexes += train[i]
        test_indexes += test[i]
    np.random.shuffle(train_indexes)
    np.random.shuffle(test_indexes)
    return train_indexes, test_indexes


def aa_and_each_accuracy(confusion_matrix):
    list_diag = np.diag(confusion_matrix)
    list_raw_sum = np.sum(confusion_matrix, axis=1)
    each_acc = np.nan_to_num(truediv(list_diag, list_raw_sum))
    average_acc = np.mean(each_acc)
    return each_acc, average_acc


def classification_map(map, ground_truth, dpi, save_path):
    fig = plt.figure(frameon=False)
    fig.set_size_inches(ground_truth.shape[1] * 2.0 / dpi, ground_truth.shape[0] * 2.0 / dpi)

    ax = plt.Axes(fig, [0., 0., 1., 1.])
    ax.set_axis_off()
    ax.xaxis.set_visible(False)
    ax.yaxis.set_visible(False)
    fig.add_axes(ax)

    ax.imshow(map)
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    fig.savefig(save_path, dpi=dpi)

    return 0


def list_to_colormap(x_list):
    y = np.zeros((x_list.shape[0], 3))
    for index, item in enumerate(x_list):
        if item == 1:
            y[index] = np.array([255, 255, 255]) / 255.
        if item == 0:
            y[index] = np.array([0, 0, 0]) / 255.
    return y




def generate_iter(TRAIN_SIZE, train_indices,  TOTAL_SIZE, total_indices, VAL_SIZE,
                  whole_data1, whole_data2, PATCH_LENGTH, padded_data1, padded_data2, INPUT_DIMENSION1,
                  INPUT_DIMENSION2, batch_size, gt):
    gt_all = gt[total_indices] - 1
    y_train = gt[train_indices] - 1

    all_data1 = select_small_cubic(TOTAL_SIZE, total_indices, whole_data1,
                                                       PATCH_LENGTH, padded_data1, INPUT_DIMENSION1)
    all_data2 = select_small_cubic(TOTAL_SIZE, total_indices, whole_data2,
                                                       PATCH_LENGTH, padded_data2, INPUT_DIMENSION2)

    train_data1 = select_small_cubic(TRAIN_SIZE, train_indices, whole_data1,
                                                         PATCH_LENGTH, padded_data1, INPUT_DIMENSION1)
    train_data2 = select_small_cubic(TRAIN_SIZE, train_indices, whole_data2,
                                                         PATCH_LENGTH, padded_data2, INPUT_DIMENSION2)

    x_train1 = train_data1.reshape(train_data1.shape[0], train_data1.shape[1], train_data1.shape[2], INPUT_DIMENSION1)

    x_train2 = train_data2.reshape(train_data2.shape[0], train_data2.shape[1], train_data2.shape[2], INPUT_DIMENSION2)
    all_data1.reshape(all_data1.shape[0], all_data1.shape[1], all_data1.shape[2], INPUT_DIMENSION1)
    all_data2.reshape(all_data2.shape[0], all_data2.shape[1], all_data2.shape[2], INPUT_DIMENSION2)

    x_val1 = all_data1[7000:9000]
    x_val2 = all_data2[7000:9000]
    y_val = gt_all[7000:9000]

    # x_val1 = all_data1
    # x_val2 = all_data2
    # y_val = gt_all



    x1_tensor_train = torch.from_numpy(x_train1).type(torch.FloatTensor)
    x2_tensor_train = torch.from_numpy(x_train2).type(torch.FloatTensor)
    y_tensor_train = torch.from_numpy(y_train).type(torch.FloatTensor)
    torch_dataset_train = Data.TensorDataset(x1_tensor_train, x2_tensor_train, y_tensor_train)


    x1_tensor_valida = torch.from_numpy(x_val1).type(torch.FloatTensor)
    x2_tensor_valida = torch.from_numpy(x_val2).type(torch.FloatTensor)
    y_tensor_valida = torch.from_numpy(y_val).type(torch.FloatTensor)
    torch_dataset_valida = Data.TensorDataset(x1_tensor_valida, x2_tensor_valida, y_tensor_valida)
    #


    all_tensor_data1 = torch.from_numpy(all_data1).type(torch.FloatTensor)
    all_tensor_data2 = torch.from_numpy(all_data2).type(torch.FloatTensor)
    all_tensor_data_label = torch.from_numpy(gt_all).type(torch.FloatTensor)
    torch_dataset_all = Data.TensorDataset(all_tensor_data1, all_tensor_data2, all_tensor_data_label)

    train_iter = Data.DataLoader(
        dataset=torch_dataset_train,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
    )
    valiada_iter = Data.DataLoader(
         dataset=torch_dataset_valida,
         batch_size=batch_size,
         shuffle=True,
         num_workers=0,
     )

    all_iter = Data.DataLoader(
        dataset=torch_dataset_all,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )
    return train_iter,  valiada_iter, all_iter, x1_tensor_train,x2_tensor_train,y_tensor_train  # , y_test


def generate_png(pred_test, gt_hsi, Dataset, total_indices):

    gt = gt_hsi.flatten()
    x_label = np.zeros(gt.shape)

    x_label[total_indices] = pred_test
    x = np.ravel(x_label)

    # print('-------Save the result in mat format--------')
    # x_re = np.reshape(x, (gt_hsi.shape[0], gt_hsi.shape[1]))
    # sio.savemat('mat/' + Dataset + '_' + '.mat', {Dataset: x_re})

    y_list = list_to_colormap(x)
    y_gt = list_to_colormap(gt)

    y_re = np.reshape(y_list, (gt_hsi.shape[0], gt_hsi.shape[1], 3))
    gt_re = np.reshape(y_gt, (gt_hsi.shape[0], gt_hsi.shape[1], 3))
    name = 'CSANetMain'
    # path = '../' + name
    path = '/kaggle/working/'
    classification_map(y_re, gt_hsi, 300,
                       path + Dataset + '.png')
    classification_map(gt_re, gt_hsi, 300,
                       path + '/classification_maps/' + Dataset + '_gt.png')
    print('------Get classification maps successful-------')



In [5]:
__all__ = ['PAM_Module', 'CAM_Module', 'CoAM_Module']


class PAM_Module(Module):
    """ Position attention module"""
    #Ref from SAGAN
    def __init__(self, in_dim):
        super(PAM_Module, self).__init__()
        self.chanel_in = in_dim

        self.query_conv = Conv2d(in_channels=in_dim, out_channels=in_dim//8, kernel_size=1)
        self.key_conv = Conv2d(in_channels=in_dim, out_channels=in_dim//8, kernel_size=1)
        self.value_conv = Conv2d(in_channels=in_dim, out_channels=in_dim, kernel_size=1)
        self.gamma = Parameter(torch.zeros(1))
        self.softmax = Softmax(dim=-1)

    def forward(self, x):
        """
            inputs :
                x : input feature maps( B X C X H X W)
            returns :
                out : attention value + input feature
                attention: B X (HxW) X (HxW)
        """
        m_batchsize, C, height, width = x.size()
        proj_query = self.query_conv(x).view(m_batchsize, -1, width * height).permute(0, 2, 1)
        proj_key = self.key_conv(x).view(m_batchsize, -1, width * height)
        energy = torch.bmm(proj_query, proj_key)
        attention = self.softmax(energy)
        proj_value = self.value_conv(x).view(m_batchsize, -1, width * height)
        out = torch.bmm(proj_value, attention.permute(0, 2, 1))
        out = out.view(m_batchsize, C, height, width)
        out = self.gamma*out + x
        return out


class CAM_Module(Module):
    """ Channel attention module"""
    def __init__(self, in_dim):
        super(CAM_Module, self).__init__()
        self.chanel_in = in_dim
        self.gamma = Parameter(torch.zeros(1))
        self.softmax = Softmax(dim=-1)

    def forward(self, x):
        """
            inputs :
                x : input feature maps( B X C X H X W)
            returns :
                out : attention value + input feature
                attention: B X C X C
        """
        m_batchsize, C, height, width = x.size()
        proj_query = x.view(m_batchsize, C, -1)
        proj_key = x.view(m_batchsize, C, -1).permute(0, 2, 1)
        energy = torch.bmm(proj_query, proj_key)
        energy_new = torch.max(energy, -1, keepdim=True)[0].expand_as(energy)-energy
        attention = self.softmax(energy_new)
        proj_value = x.view(m_batchsize, C, -1)
        out = torch.bmm(attention, proj_value)
        out = out.view(m_batchsize, C, height, width)
        out = self.gamma*out + x
        return out

class CoAM_Module(Module):
    """ Correlation attention module"""

    def __init__(self, in_dim):
        super(CoAM_Module, self).__init__()
        self.chanel_in = in_dim
        self.gamma = Parameter(torch.zeros(1))
        self.softmax = Softmax(dim=-1)
    def forward (self, x1, x2):
        """
            inputs :
                x1 : input feature maps1( B X C X H X W)
                x2: input feature maps2( B X C X H X W)
            returns :
                out : attention value2 + input feature1
                attention: B X C X C
        """
        m_batchsize, C, height, width = x1.size()
        proj_query = x2.view(m_batchsize, C, -1)
        proj_key = x2.view(m_batchsize, C, -1).permute(0, 2, 1)
        energy = torch.bmm(proj_query, proj_key)
        energy_new = torch.max(energy, -1, keepdim=True)[0].expand_as(energy)-energy
        attention = self.softmax(energy_new)
        proj_value = x1.view(m_batchsize, C, -1)

        out = torch.bmm(attention, proj_value)
        out = out.view(m_batchsize, C, height, width)

        out = self.gamma*out + x1
        return out

In [6]:
class FeatureFusionModule(nn.Module):
    def __init__(self, input_channels):
        super(FeatureFusionModule, self).__init__()

        # Step 1: Feature Summation
        # self.feature_sum = nn.Conv2d(input_channels, 1, kernel_size=1)

        # Step 2: Channel Compression

        self.conv_1 = nn.Conv2d(input_channels, 128, kernel_size=3,padding=1)
        self.conv_2 = nn.Conv2d(input_channels, 128, kernel_size=3,padding=1)
        self.conv_3 = nn.Conv2d(128, 64, kernel_size=3,padding=1)
        self.channel_compression = nn.Conv2d(1, 2, kernel_size=3, padding=1)

    def forward(self, feature1, feature2):
        # Step 1: Feature Summation

        # Step 2: Channel Compression
        f = self.conv_1(feature1+ feature2)+self.conv_2(feature1 + feature2)
        f = self.conv_3(f)
        f = f.mean(dim=1, keepdim=True)
        compressed_feature = self.channel_compression(f)

        # Step 3: Matrix Generation and Softmax Normalization

        weight = F.softmax(compressed_feature, dim=1)
        a = torch.unsqueeze(weight[:,0,:,:],1)
        b = torch.unsqueeze(weight[:,1,:,:],1)

        # Step 4: Element-wise Multiplication with Original Features
        output1 = (feature1+ feature2) * a + feature1
        output2 = (feature1+ feature2) * b + feature2

        return output1, output2


class FeatureMultiscale(nn.Module):
    def __init__(self, input_channels):
        super(FeatureMultiscale, self).__init__()

        self.conv_4 = nn.Conv2d(128, 64, kernel_size=3,padding=1)
        self.conv_5 = nn.Conv2d(64, 32, kernel_size=3,padding=1)

        self.conv_1 = nn.Conv2d(input_channels, 128, kernel_size=3,padding=1)
        self.conv_2 = nn.Conv2d(input_channels, 128, kernel_size=3,padding=1)
        self.conv_3 = nn.Conv2d(input_channels, 128, kernel_size=3,padding=1)
        self.GAP = nn.AdaptiveAvgPool2d((1,1))
        self.channel_compression = nn.Conv2d(128, 64, kernel_size=1)
        self.channel_expand = nn.Conv2d(64, input_channels*3, kernel_size=1)

    def forward(self, feature1, feature2,feature3):
        # Step 1: Feature Summation
        f = feature1 + feature2 + feature3
        f = self.conv_1(f)+self.conv_2(f)+self.conv_3(f)
        # f = self.conv_5(self.conv_4(f))
        f = F.adaptive_avg_pool2d(f, (1, 1))
        compressed_feature = self.channel_compression(f)
        expand_feature = self.channel_expand(compressed_feature)
        b,w,x,y = feature1.size()
        expand_feature = expand_feature.view(b,w,3,1,1)

        weight = F.softmax(expand_feature, dim=2)
        a =weight[:,:,0,:,:]
        b =weight[:,:,1,:,:]
        c =weight[:,:,2,:,:]

        output1 = feature1 * a
        output2 = feature2 * b
        output3 = feature2 * c

        return output1+output2+output3

class deeplab_V2(nn.Module):
    def __init__(self):
        super(deeplab_V2, self).__init__()
        self.fusion1 = FeatureFusionModule(128)
        self.fusion2= FeatureFusionModule(128)
        self.multiattentionfusion1= FeatureMultiscale(128)
        self.multiattentionfusion2= FeatureMultiscale(128)
        # self.fusion_add_1= FeatureFusionModule2(64)
        # self.fusion_add_2= FeatureFusionModule2(64)
        # self.fusion_add_3= FeatureFusionModule2(64)
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels=155, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1, ceil_mode=True),

        )
        '''
        '''
        self.conv2 = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=128, out_channels=512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            # nn.MaxPool2d(kernel_size=3, stride=2, padding=1, ceil_mode=True),

        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),#![](classification_maps/IN_gt.png)
            nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1, ceil_mode=True),

        )
        self.conv4 = nn.Sequential(
            nn.Conv2d(in_channels=256, out_channels=512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1, ceil_mode=True),

        )
        self.conv5 = nn.Sequential(
            nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, dilation=2, padding=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, dilation=2, padding=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, dilation=2, padding=2),
            nn.ReLU(inplace=True),

        )  #进行卷积操作


        inter_channels = 512 // 4###################################################
        self.conv5a = nn.Sequential(nn.Conv2d(512, inter_channels, 3, padding=1, bias=False),

                                    nn.ReLU())

        self.conv5c = nn.Sequential(nn.Conv2d(512, inter_channels, 3, padding=1, bias=False),

                                    nn.ReLU())

        self.sa = PAM_Module(inter_channels)####
        self.sc = CAM_Module(inter_channels)
        self.sco = CoAM_Module(inter_channels)###
        self.conv51 = nn.Sequential(nn.Conv2d(inter_channels, inter_channels, 3, padding=1, bias=False),

                                    nn.ReLU())
        self.conv52 = nn.Sequential(nn.Conv2d(inter_channels, inter_channels, 3, padding=1, bias=False),

                                    nn.ReLU())


        self.conv8 = nn.Sequential(nn.Dropout2d(0.1, False), nn.Conv2d(128, 128,  3, padding=1))
        # self.conv9 = nn.Sequential(nn.Dropout2d(0.1, False), nn.Conv2d(128, 128,  3, padding=1))


    def forward(self, x1, x2):
        x1 = self.conv1(x1)
        x1 = self.conv2(x1)
        x2 = self.conv1(x2)
        x2 = self.conv2(x2)


        feat11 = self.conv5a(x1)
        feat21 = self.conv5a(x2)

        feat12 = self.conv5c(x1)
        feat22 = self.conv5c(x2)

        feat11, feat21 = self.fusion1(feat11, feat21)
        feat12, feat22 = self.fusion2(feat12, feat22)
######################################################################
        #生成sa attention
        sa_feat1 = self.sa(feat11)
        sa_feat2 = self.sa(feat21)

        sa_conv1 = self.conv51(sa_feat1)
        sa_conv2 = self.conv51(sa_feat2)

        # 生成sc attention
        sc_feat1 = self.sc(feat12)
        sc_feat2 = self.sc(feat22)

        sc_conv1 = self.conv52(sc_feat1)
        sc_conv2 = self.conv52(sc_feat2)


        #生成temporal
        sco_conv1 = self.sco(feat11,feat12)
        sco_conv1 = self.conv51(sco_conv1)
        sco_conv2 = self.sco(feat21,feat22)
        sco_conv2 = self.conv51(sco_conv2)


        # sco_conv1,sco_conv2 = self.fusion(sco_conv1,sco_conv2)
        feat_sum1 = self.multiattentionfusion1(sa_conv1,sc_conv1,sco_conv1)
        feat_sum2 = self.multiattentionfusion2(sa_conv2,sc_conv2,sco_conv2)
        # feat_sum1 = sa_conv1 + sc_conv1 + 0.3*sco_conv1
        # feat_sum2 = sa_conv2 + sc_conv2 + 0.3*sco_conv2


        sasc_output1 = self.conv8(feat_sum1)
        sasc_output2 = self.conv8(feat_sum2)



        return sasc_output1, sasc_output2


class SiameseNet(nn.Module):
    def __init__(self, norm_flag='l2'):
        super(SiameseNet, self).__init__()
        self.CNN = deeplab_V2()

        if norm_flag == 'l2':
            self.norm = F.normalize  #F.normalize对输入的数据（tensor）进行指定维度的L2_norm运算
        if norm_flag == 'exp':
            self.norm = nn.Softmax2d()

    def forward(self, t0, t1):
        t0 = t0.float()
        t1 = t1.float()
        out_t0_embedding, out_t1_embedding, = self.CNN(t0, t1)
        return out_t0_embedding, out_t1_embedding


class Classifier(nn.Module):
    def __init__(self):
        super(Classifier, self).__init__()
        self.conv1 = nn.Conv2d(128, 64, 3, padding=1, padding_mode='reflect', bias=False)  # [64, 24, 24]
        self.bat1 = nn.BatchNorm2d(64)#
        self.reli1 = nn.LeakyReLU(0.2)
        self.pool1 = nn.MaxPool2d(2)

        self.conv2 = nn.Conv2d(64, 32, 3, padding=1, padding_mode='reflect', bias=False)
        self.bat2 = nn.BatchNorm2d(32)
        self.reli2 = nn.LeakyReLU(0.2)

    def forward(self, x):
        con1 = self.conv1(x)
        ba1 = self.bat1(con1)
        re1 = self.reli1(ba1)
        po1 = self.pool1(re1)
        con2 = self.conv2(po1)
        ba2 = self.bat2(con2)
        re2 = self.reli2(ba2)

        return re2


class ChangeNet(nn.Module):
    def __init__(self):
        super(ChangeNet, self).__init__()
        self.singlebrach = Classifier()# re2

    def forward(self, t0, t1):

        indata = t0 - t1
        c3 = self.singlebrach(indata)

        return c3


class BSNET_Conv(nn.Module):

    def __init__(self, ):
        super(BSNET_Conv, self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(155, 64, (3, 3), 1, 0),
            nn.ReLU(True))

        self.conv1_1 = nn.Sequential(
            nn.Conv2d(155, 128, (3, 3), 1, 0),
            nn.ReLU(True))
        self.conv1_2 = nn.Sequential(
            nn.Conv2d(128, 64, (3, 3), 1, 0),
            nn.ReLU(True))

        self.deconv1_2 = nn.Sequential(
            nn.ConvTranspose2d(64, 64, (3, 3), 1, 0),
            nn.ReLU(True))

        self.deconv1_1 = nn.Sequential(
            nn.ConvTranspose2d(64, 128, (3, 3), 1, 0),
            nn.ReLU(True))

        self.conv2_1 = nn.Sequential(
            nn.Conv2d(128, 155, (1, 1), 1, 0),
            nn.Sigmoid())

        self.fc1 = nn.Sequential(
            nn.Linear(64, 128),
            nn.ReLU(True))
        self.fc2 = nn.Sequential(
            nn.Linear(128, 155),
            nn.Sigmoid())

    def GlobalPool(self, feature_size):
        return nn.AvgPool2d(kernel_size=feature_size)

    def BAM(self, x):
        x = self.conv1(x)
        gp = self.GlobalPool(x.shape[2])
        x = gp(x)
        x = x.view(-1, 64)
        x = self.fc1(x)
        x = self.fc2(x)
        x = x.view(-1, 1, 1, 155)
        x = x.permute(0, 3, 1, 2)
        return x

    def RecNet(self, x):
        x = self.conv1_1(x)
        x = self.conv1_2(x)
        x = self.deconv1_2(x)
        x = self.deconv1_1(x)
        x = self.conv2_1(x)
        return x

    def forward(self, x):
        BRW = self.BAM(x)
        new_x = x * BRW
        ret = self.RecNet(new_x)

        return new_x,F.mse_loss(ret,x),BRW

class Finalmodel(nn.Module):#######################nn,Module################################################################
    def __init__(self):
        super(Finalmodel, self).__init__()
        self.siamesnet = SiameseNet()
        self.chnet = ChangeNet() #c3
        self.fc = nn.Sequential(
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )

        self.conv_con = nn.Sequential(
            nn.Conv2d(32, 64, (1, 1), 1, 0),
            nn.ReLU(True)
        )


        self.fc1 = nn.Sequential(
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 2)
        )

        self.maxpool = nn.MaxPool2d(2)

        self.bf = BSNET_Conv()
        self.conv_re = nn.Sequential(
            nn.Conv2d(155, 128, (3, 3), 1, 0),
            nn.ReLU(True)
        )
        self.classifer = Classifier()

        self.fc_con = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU()
        )

    def forward(self, t0, t1):
        t0 = t0.permute(0, 3, 1, 2) #换个顺序 0123-----0312
        t1 = t1.permute(0, 3, 1, 2)

        t_diff = t1-t0
        t_diff, recon_error, BRW = self.bf(t_diff)

        out1 = self.classifer(self.conv_re(t_diff))
        out1 = self.maxpool(out1)
        out1 = out1.view(out1.size(0), -1)
        out1 = self.fc1(out1)
        out1 = F.sigmoid(out1)

        t0=t0*BRW
        t1=t1*BRW

        x1, x2 = self.siamesnet(t0, t1)
        out = self.chnet(x1, x2) ###转化为32通道的2-D特征
        out = self.maxpool(out)

        # contrastive_feature = self.fc_con(out.view(out.size(0), -1))
        # contrastive_feature = contrastive_feature.view(contrastive_feature.size(0),2, -1)

        feature = self.conv_con(out)
        feature = feature.view(feature.size(0), -1)
        contrastive_feature = self.fc_con(feature)
        #
        # contrastive_feature = self.fc(out)
        out = out.view(out.size(0), -1)
        out = self.fc(out)
        out = F.sigmoid(out)


        return out,t0,t1,recon_error,out1,contrastive_feature

In [7]:
def use_svg_display():
    # 用矢量图显示
    display.set_matplotlib_formats('svg')


def set_figsize(figsize=(3.5, 2.5)):
    # use_svg_display()
    # 设置图的尺寸
    plt.rcParams['figure.figsize'] = figsize


def data_iter(batch_size, features, labels):
    num_examples = len(features)
    indices = list(range(num_examples))
    random.shuffle(indices)
    for i in range(0, num_examples, batch_size):
        j = torch.LongTensor(
            indices[i:min(i + batch_size, num_examples)])  # 类型转换为long和tensor
        yield features.index_select(0, j), labels.index_select(
            0, j)  # index_select (dim, index_list/int), dim代表那个轴 输出与输入在轴的维度上相同


def sgd(params, lr, batch_size):  # 本函数已保存在d2lzh_pytorch包中方便以后使用
    for param in params:
        param.data -= lr * param.grad / batch_size  # 注意这里更改param时用的param.data


def show_fashion_mnist(images, labels):
    _, figs = plt.subplots(1, len(images), figsize=(12, 12))
    for f, img, lbl in zip(figs, images, labels):
        f.imshow(img.view((28, 28)).numpy())
        f.set_title(lbl)
        f.axes.get_xaxis().set_visible(False)
        f.axes.get_yaxis().set_visible(False)
    plt.show()


def get_fashion_mnist_labels(labels):
    text_labels = [
        't-shirt', 'trouser', 'pullover', 'dress', 'coat', 'sandal', 'shirt',
        'sneaker', 'bag', 'ankle boot'
    ]
    return [text_labels[int(i)] for i in labels]


def load_data_fashion_mnist(batch_size):
    if sys.platform.startswith('win'):
        num_workers = 0
    else:
        num_workers = 4
    mnist_train = torchvision.datasets.FashionMNIST(
        root="./DataSets/FashionMNIST",
        train=True,
        download=True,
        transform=transforms.ToTensor())

    mnist_test = torchvision.datasets.FashionMNIST(
        root="./DataSets/FashionMNIST",
        train=False,
        download=True,
        transform=transforms.ToTensor())
    train_iter = torch.utils.data.DataLoader(mnist_train,
                                             batch_size=batch_size,
                                             shuffle=True,
                                             num_workers=num_workers)
    test_iter = torch.utils.data.DataLoader(mnist_test,
                                            batch_size=batch_size,
                                            shuffle=False,
                                            num_workers=num_workers)
    return train_iter, test_iter


def evaluate_accuracy(data_iter, net):
    acc_sum, n = 0.0, 0
    for x, y in data_iter:
        acc_sum += (net(x).argmax(dim=1)==y).float().sum().item()
        n += y.shape[0]
    return acc_sum / n


def train_ch3(net, train_iter, test_iter, loss, num_epochs, batch_size, params=None, lr=None, optimizer=None):
    for epoch in range(num_epochs):
        train_l_sum, train_acc_sum, n = 0.0, 0.0, 0
        for x, y in train_iter:
            y_hat = net(x)
            l = loss(y_hat, y).sum()
            if optimizer is not None:
                optimizer.zero_grad()
            elif params is not None and params[0].grad is not None:
                for param in params:
                    param.grad.data.zero_()
            l.backward()
            if optimizer is None:
                sgd(params, lr, batch_size)
            else:
                optimizer.step()
            train_l_sum += l.item()
            train_acc_sum += (y_hat.argmax(dim=1) == y).sum().item()
            n += y.shape[0]
        test_acc = evaluate_accuracy(test_iter, net)
        print('epoch %d, loss %.4f, train acc %.3f, test acc %.3f'
              % (epoch + 1, train_l_sum / n, train_acc_sum / n, test_acc))

In [8]:
class BalSCL(nn.Module):
    def __init__(self, cls_num_list=None, temperature=0.1):
        super(BalSCL, self).__init__()
        self.temperature = temperature
        self.cls_num_list = cls_num_list

    def forward(self, centers1, features, targets):

        device = (torch.device('cuda') if features.is_cuda else torch.device('cpu'))
        batch_size = features.shape[0]

        # 将样本标签重塑为二维形状
        targets = targets.contiguous().view(-1, 1).long()

        # 构建包含类别中心标签的扩展 targets
        targets_centers = torch.arange(len(self.cls_num_list), device=device).view(-1,1).long()
        targets = torch.cat([targets, targets_centers], dim=0) ###复制一遍标签，再把targets center贴到后面

        # 计算每个类别在批次中的样本数
        batch_cls_count = torch.eye(len(self.cls_num_list), dtype=torch.float32)[targets].sum(dim=0).squeeze().long()

        # 构建掩码，用于区分正负样本对
        mask = torch.eq(targets[: batch_size], targets.T).float().to(device)
        logits_mask = torch.scatter(
            torch.ones_like(mask),
            1,
            torch.arange(batch_size * 1).view(-1, 1).to(device),
            0
        )
        mask = mask * logits_mask

        # 将特征向量与类别中心连接，用于计算相似度矩阵
        # features = torch.cat(torch.unbind(features, dim=1), dim=0)
        features = torch.cat([features, centers1], dim=0)
        logits = features[: batch_size].mm(features.T)
        logits = torch.div(logits, self.temperature)

        # 为数值稳定性添加修正
        logits_max, _ = torch.max(logits, dim=1, keepdim=True)
        logits = logits - logits_max.detach()

        # 计算每个样本的 softmax 权重
        exp_logits = torch.exp(logits) * logits_mask
        per_ins_weight = torch.tensor([batch_cls_count[i] for i in targets], device=device).view(1, -1).expand(
            1 * batch_size, 1 * batch_size + len(self.cls_num_list)) - mask

        exp_logits_sum = exp_logits.div(per_ins_weight).sum(dim=1, keepdim=True)

        # 计算对数概率，用于计算平均对数概率
        log_prob = logits - torch.log(exp_logits_sum)
        mean_log_prob_pos = (mask * log_prob).sum(1) / mask.sum(1)

        # 计算损失
        loss = - mean_log_prob_pos
        loss = loss.view(batch_size).mean()
        return loss


def evaluate_accuracy(data_iter, net, loss, device):
    acc_sum, n = 0.0, 0
    with torch.no_grad():
        for X1, X2, y in data_iter:
            test_l_sum, test_num = 0, 0
            X1 = X1.to(device)
            X2 = X2.to(device)
            y = y.to(device)
            net.eval() # 评估模式, 这会关闭dropout
            y_hat,_,_,_,_,_= net(X1,X2)
            l = loss(y_hat, y.long())
            # l=l1
            # l2 = loss2(out1, out2, y.long()-1)
            # l = 0.8 * l1+ 0.2 * l2
            acc_sum += (y_hat.argmax(dim=1).long() == y.long().to(device)).sum().cpu().item()
            test_l_sum += l
            test_num += 1
            net.train() # 改回训练模式
            n += y.shape[0]
    return [acc_sum / n, test_l_sum] # / test_num]

def train(net, train_iter, valida_iter, loss,  optimizer, optimizer_bs, optimizer_con,device, epochs,train_all_x1,train_all_x2,train_all_label):
    loss_list = [100]
    early_epoch = 0
    best = 0
    net = net.to(device)
    print("training on ", device)
    start = time.time()
    train_loss_list = []
    valida_loss_list = []
    train_acc_list = []
    valida_acc_list = []
    newnet = []
    contrastive_loss = BalSCL(cls_num_list=[0,1])
    torch.autograd.set_detect_anomaly = True
    for epoch in range(epochs):
        ########寻找原型
        train_acc_sum, n = 0.0, 0
        time_epoch = time.time()
        lr_adjust = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, 15, eta_min=0.0, last_epoch=-1)
        for X1, X2, y in train_iter:
            batch_count, train_l_sum = 0, 0
            X1 = X1.to(device)
            X2 = X2.to(device)
            y = y.to(device)
            y_hat, m0,m1,rec_error,out,contrastive_feature = net(X1, X2)


            l= loss(y_hat, y.long())

            optimizer.zero_grad()
            l.backward()
            optimizer.step()

            train_l_sum += l.cpu().item()
            train_acc_sum += (y_hat.argmax(dim=1).long() == y.long()).sum().cpu().item()
            n += y.shape[0]
            batch_count += 1
        lr_adjust.step(epoch)
        valida_acc, valida_loss = evaluate_accuracy(valida_iter, net, loss, device)
        loss_list.append(valida_loss)
        # if valida_acc >= best:
        #     best = valida_acc
        torch.save(net,'best.pth')

        print(best)
        # 绘图部分
        train_loss_list.append(train_l_sum) # / batch_count)
        train_acc_list.append(train_acc_sum / n)
        valida_loss_list.append(valida_loss)
        valida_acc_list.append(valida_acc)

        print('epoch %d, train loss %.6f, train acc %.3f, valida loss %.6f, valida acc %.3f, time %.1f sec'
                % (epoch + 1, train_l_sum / batch_count, train_acc_sum / n, valida_loss, valida_acc, time.time() - time_epoch))

        PATH = "./net_DBA.pt"
        # if loss_list[-1] <= 0.01 and valida_acc >= 0.95:
        #     torch.save(net.state_dict(), PATH)
        #     break



    set_figsize()
    plt.figure(figsize=(8, 8.5))
    train_accuracy = plt.subplot(221)
    train_accuracy.set_title('train_accuracy')
    plt.plot(np.linspace(1, epoch, len(train_acc_list)), train_acc_list, color='green')
    plt.xlabel('epoch')
    plt.ylabel('train_accuracy')
    # train_acc_plot = np.array(train_acc_plot)
    # for x, y in zip(num_epochs, train_acc_plot):
    #    plt.text(x, y + 0.05, '%.0f' % y, ha='center', va='bottom', fontsize=11)

    test_accuracy = plt.subplot(222)
    test_accuracy.set_title('valida_accuracy')
    plt.plot(np.linspace(1, epoch, len(valida_acc_list)), valida_acc_list, color='deepskyblue')
    plt.xlabel('epoch')
    plt.ylabel('test_accuracy')
    # test_acc_plot = np.array(test_acc_plot)
    # for x, y in zip(num_epochs, test_acc_plot):
    #   plt.text(x, y + 0.05, '%.0f' % y, ha='center', va='bottom', fontsize=11)

    loss_sum = plt.subplot(223)
    loss_sum.set_title('train_loss')
    plt.plot(np.linspace(1, epoch, len(valida_acc_list)), valida_acc_list, color='red')
    plt.xlabel('epoch')
    plt.ylabel('train loss')
    # ls_plot = np.array(ls_plot)

    test_loss = plt.subplot(224)
    test_loss.set_title('valida_loss')
    #plt.plot(np.linspace(1, epoch, len(valida_loss_list)), valida_loss_list, color='gold')
    #plt.xlabel('epoch')
    #plt.ylabel('valida loss')
    # ls_plot = np.array(ls_plot)

    #plt.show()
    print('epoch %d, loss %.4f, train acc %.3f, time %.1f sec'
            % (epoch + 1, train_l_sum / batch_count, train_acc_sum / n, time.time() - start))

    return newnet


In [9]:
# Farmland

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

# for Monte Carlo runs
seeds = [1331, 1332, 1333, 1334, 1335, 1336, 1337, 1338, 1339, 1340, 1341]
ensemble = 1

day = datetime.datetime.now()
day_str = day.strftime('%m_%d_%H_%M')

print('-----Importing Dataset-----')


global Dataset  # UP,IN,KSC
Dataset = 'china'
T11, T22, gt_hsi, TOTAL_SIZE, TRAIN_SIZE, VALIDATION_SPLIT = load_dataset()
print(T11.shape)

image_x, image_y, BAND1 = T11.shape



data1 = T11.reshape(np.prod(T11.shape[:2]), np.prod(T11.shape[2:]))
data2 = T22.reshape(np.prod(T22.shape[:2]), np.prod(T22.shape[2:]))
gt = gt_hsi.reshape(np.prod(gt_hsi.shape[:2]),)
CLASSES_NUM = 2


print('-----Importing Setting Parameters-----')
ITER = 1
PATCH_LENGTH = 4
# number of training samples per class
# lr, num_epochs, batch_size = 0.00002, 100, 32
lr, num_epochs, batch_size = 0.00002, 100,32
loss = torch.nn.CrossEntropyLoss()


img_rows = 2*PATCH_LENGTH+1
img_cols = 2*PATCH_LENGTH+1
img_channels1 = T11.shape[2]
img_channels2 = T22.shape[2]
INPUT_DIMENSION1 = T11.shape[2]
INPUT_DIMENSION2 = T22.shape[2]
ALL_SIZE = T11.shape[0] * T22.shape[1]
VAL_SIZE = int(TRAIN_SIZE)/2



KAPPA = []
OA = []
AA = []
TRAINING_TIME = []
TESTING_TIME = []
ELEMENT_ACC = np.zeros((ITER, CLASSES_NUM))

data1 = preprocessing.scale(data1)
data2 = preprocessing.scale(data2)

data1_ = data1.reshape(T11.shape[0], T11.shape[1], T11.shape[2])
whole_data1 = data1_
data2_ = data2.reshape(T22.shape[0], T22.shape[1], T22.shape[2])
whole_data2 = data2_


padded_data1 = np.lib.pad(whole_data1, ((PATCH_LENGTH, PATCH_LENGTH), (PATCH_LENGTH, PATCH_LENGTH), (0, 0)),
                         'constant', constant_values=0)
padded_data2 = np.lib.pad(whole_data2, ((PATCH_LENGTH, PATCH_LENGTH), (PATCH_LENGTH, PATCH_LENGTH), (0, 0)),
                         'constant', constant_values=0)

for index_iter in range(ITER):
    print(index_iter)
    net = Finalmodel()
    params_to_optimize=[
    {"params": net.siamesnet.parameters()},  # 其他网络部分的参数
    {"params": net.chnet.parameters()},  # 其他网络部分的参数
    {"params": net.fc.parameters()},  # 其他网络部分的参数
    {"params": net.bf.parameters()},  # self.bf = BSNET_Conv() 的参数
    {"params": net.conv_con.parameters()},
    {"params": net.fc_con.parameters()}
    ]
    # optimizer = optim.Adam(params_to_optimize, lr=lr)  # , weight_decay=0.0001)
    optimizer = optim.Adam(params_to_optimize[0:2], lr=lr)  # , weight_decay=0.0001)
    optimizer_bs = optim.Adam([{'params': net.bf.parameters()},], lr=0.00001)  # , weight_decay=0.0001)
    optimizer_con = optim.Adam(params_to_optimize[4:5], lr=0.00002)  # , weight_decay=0.0001)
    time_1 = int(time.time())
    np.random.seed(seeds[index_iter])
    train_indices, test_indices = sampling(VALIDATION_SPLIT, gt)
    _, total_indices = sampling(1, gt)

    TRAIN_SIZE = len(train_indices)
    print('Train size: ', TRAIN_SIZE)
    VAL_SIZE = int(TRAIN_SIZE)
    print('Validation size: ', VAL_SIZE)

    print('-----Selecting Small Pieces from the Original Cube Data-----')


    train_iter,  valida_iter, all_iter,train_all_1,train_all_2,train_all_label = generate_iter(TRAIN_SIZE, train_indices, TOTAL_SIZE, total_indices, VAL_SIZE,
                  whole_data1, whole_data2, PATCH_LENGTH, padded_data1, padded_data2, INPUT_DIMENSION1, INPUT_DIMENSION2, batch_size, gt)
    #R
    #
    train(net, train_iter, valida_iter, loss, optimizer,optimizer_bs, optimizer_con, device, num_epochs,train_all_1,train_all_2,train_all_label)

    # tic2 = time.clock()
    newnet = Finalmodel()

    try:
        # Try loading as state_dict first
        state_dict = torch.load("best.pth", map_location=device, weights_only=False)
        if isinstance(state_dict, dict):
            newnet.load_state_dict(state_dict)
        else:
            # If it's a model object, extract state_dict
            newnet.load_state_dict(state_dict.state_dict())
    except:
        # If all else fails, load the entire model
        newnet = torch.load("best.pth", map_location=device, weights_only=False)
    
    newnet.to(device)
    newnet.eval()

    
    pred_test_fdssc = []
    with torch.no_grad():
        for X1, X2, y in all_iter:
            X1 = X1.to(device)
            X2 = X2.to(device)
            y_hat, _, _, _, _, _ = newnet(X1, X2)
            pred_test_fdssc.extend(np.array(y_hat.cpu().argmax(axis=1)))
    # toc2 = time.clock()

    Counter(pred_test_fdssc)
    gt_test = gt[total_indices]-1
    # gt_test = gt_test1[:-VAL_SIZE]
    # print('pre', np.unique(pred_test_fdssc))
    # print('gt_test', np.unique(gt_test[:-VAL_SIZE]))

    overall_acc_fdssc = metrics.accuracy_score(pred_test_fdssc, gt_test)
    confusion_matrix_fdssc = metrics.confusion_matrix(pred_test_fdssc, gt_test)
    # print(confusion_matrix_fdssc)
    each_acc_fdssc, average_acc_fdssc = aa_and_each_accuracy(confusion_matrix_fdssc)
    kappa = metrics.cohen_kappa_score(pred_test_fdssc, gt_test)

    # print("testing time:",toc2-tic2)
    savemat("river.mat", mdict={'result': pred_test_fdssc})
    KAPPA.append(kappa)
    OA.append(overall_acc_fdssc)
    AA.append(average_acc_fdssc)
    ELEMENT_ACC[index_iter, :] = each_acc_fdssc


    def calculate_metrics(confusion_matrix):
        # 从混淆矩阵中提取TP, TN, FP, FN
        TP = confusion_matrix[1, 1]
        TN = confusion_matrix[0, 0]
        FP = confusion_matrix[0, 1]
        FN = confusion_matrix[1, 0]

        # 计算Precision
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0

        # 计算Recall
        recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0

        # 计算F1 Score
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

        # 计算IoU (Intersection over Union)
        iou = TP / (TP + FP + FN) if (TP + FP + FN) > 0 else 0.0

        return precision, recall, f1, iou


    precision, recall, f1, iou = calculate_metrics(confusion_matrix_fdssc)

    print("-------- Training Finished-----------")
    print('OA:',OA)
    print('AA:',AA)
    print('Kappa:',KAPPA)
    print('OA_UN',each_acc_fdssc)
    print('precision',precision)
    print('recall',recall)
    print('f1',f1)
    print('iou',iou)


    generate_png(pred_test_fdssc, gt_hsi-1, 'China', total_indices)




cuda:0
-----Importing Dataset-----


KeyError: 'T1'

In [ ]:
def load_dataset():
    # Load Bay Area temporal images and ground truth
    T1_ori = sio.loadmat('/kaggle/input/sst-change-detection-dataset/ChangeDetectionDataset-master-72b0263758e1f4dc04769bac5a688f951f5ce23a/bayArea/mat/Bay_Area_2013.mat')
    T2_ori = sio.loadmat('/kaggle/input/sst-change-detection-dataset/ChangeDetectionDataset-master-72b0263758e1f4dc04769bac5a688f951f5ce23a/bayArea/mat/Bay_Area_2015.mat')
    mat_gt = sio.loadmat('/kaggle/input/sst-change-detection-dataset/ChangeDetectionDataset-master-72b0263758e1f4dc04769bac5a688f951f5ce23a/bayArea/mat/bayArea_gtChanges2.mat.mat')

    # Replace keys with correct variable names inside .mat files
    TT1 = T1_ori['HypeRvieW']        # e.g., T1: 3D array of size (H, W, Bands)
    TT2 = T2_ori['HypeRvieW']
    gt_hsi = mat_gt['HypeRvieW']     # Ground truth: 2D array of size (H, W)

    TOTAL_SIZE = TT1.shape[0] * TT1.shape[1]
    VALIDATION_SPLIT = 0.97
    TRAIN_SIZE = math.ceil(TOTAL_SIZE * VALIDATION_SPLIT)

    return TT1, TT2, gt_hsi, TOTAL_SIZE, TRAIN_SIZE, VALIDATION_SPLIT


In [ ]:
# Hermiston
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

# for Monte Carlo runs
seeds = [1331, 1332, 1333, 1334, 1335, 1336, 1337, 1338, 1339, 1340, 1341]
ensemble = 1

day = datetime.datetime.now()
day_str = day.strftime('%m_%d_%H_%M')

print('-----Importing Dataset-----')


global Dataset  # UP,IN,KSC
Dataset = 'hermiston'
T11, T22, gt_hsi, TOTAL_SIZE, TRAIN_SIZE, VALIDATION_SPLIT = load_dataset()
print(T11.shape)

image_x, image_y, BAND1 = T11.shape



data1 = T11.reshape(np.prod(T11.shape[:2]), np.prod(T11.shape[2:]))
data2 = T22.reshape(np.prod(T22.shape[:2]), np.prod(T22.shape[2:]))
gt = gt_hsi.reshape(np.prod(gt_hsi.shape[:2]),)
CLASSES_NUM = 2


print('-----Importing Setting Parameters-----')
ITER = 1
PATCH_LENGTH = 4
# number of training samples per class
# lr, num_epochs, batch_size = 0.00002, 100, 32
lr, num_epochs, batch_size = 0.00002, 100,32
loss = torch.nn.CrossEntropyLoss()


img_rows = 2*PATCH_LENGTH+1
img_cols = 2*PATCH_LENGTH+1
img_channels1 = T11.shape[2]
img_channels2 = T22.shape[2]
INPUT_DIMENSION1 = T11.shape[2]
INPUT_DIMENSION2 = T22.shape[2]
ALL_SIZE = T11.shape[0] * T22.shape[1]
VAL_SIZE = int(TRAIN_SIZE)/2



KAPPA = []
OA = []
AA = []
TRAINING_TIME = []
TESTING_TIME = []
ELEMENT_ACC = np.zeros((ITER, CLASSES_NUM))

data1 = preprocessing.scale(data1)
data2 = preprocessing.scale(data2)

data1_ = data1.reshape(T11.shape[0], T11.shape[1], T11.shape[2])
whole_data1 = data1_
data2_ = data2.reshape(T22.shape[0], T22.shape[1], T22.shape[2])
whole_data2 = data2_


padded_data1 = np.lib.pad(whole_data1, ((PATCH_LENGTH, PATCH_LENGTH), (PATCH_LENGTH, PATCH_LENGTH), (0, 0)),
                         'constant', constant_values=0)
padded_data2 = np.lib.pad(whole_data2, ((PATCH_LENGTH, PATCH_LENGTH), (PATCH_LENGTH, PATCH_LENGTH), (0, 0)),
                         'constant', constant_values=0)

for index_iter in range(ITER):
    print(index_iter)
    net = Finalmodel()
    params_to_optimize=[
    {"params": net.siamesnet.parameters()},  # 其他网络部分的参数
    {"params": net.chnet.parameters()},  # 其他网络部分的参数
    {"params": net.fc.parameters()},  # 其他网络部分的参数
    {"params": net.bf.parameters()},  # self.bf = BSNET_Conv() 的参数
    {"params": net.conv_con.parameters()},
    {"params": net.fc_con.parameters()}
    ]
    # optimizer = optim.Adam(params_to_optimize, lr=lr)  # , weight_decay=0.0001)
    optimizer = optim.Adam(params_to_optimize[0:2], lr=lr)  # , weight_decay=0.0001)
    optimizer_bs = optim.Adam([{'params': net.bf.parameters()},], lr=0.00001)  # , weight_decay=0.0001)
    optimizer_con = optim.Adam(params_to_optimize[4:5], lr=0.00002)  # , weight_decay=0.0001)
    time_1 = int(time.time())
    np.random.seed(seeds[index_iter])
    train_indices, test_indices = sampling(VALIDATION_SPLIT, gt)
    _, total_indices = sampling(1, gt)

    TRAIN_SIZE = len(train_indices)
    print('Train size: ', TRAIN_SIZE)
    VAL_SIZE = int(TRAIN_SIZE)
    print('Validation size: ', VAL_SIZE)

    print('-----Selecting Small Pieces from the Original Cube Data-----')


    train_iter,  valida_iter, all_iter,train_all_1,train_all_2,train_all_label = generate_iter(TRAIN_SIZE, train_indices, TOTAL_SIZE, total_indices, VAL_SIZE,
                  whole_data1, whole_data2, PATCH_LENGTH, padded_data1, padded_data2, INPUT_DIMENSION1, INPUT_DIMENSION2, batch_size, gt)
    #R
    #
    train(net, train_iter, valida_iter, loss, optimizer,optimizer_bs, optimizer_con, device, num_epochs,train_all_1,train_all_2,train_all_label)

    # tic2 = time.clock()
    newnet = Finalmodel()

    try:
        # Try loading as state_dict first
        state_dict = torch.load("best.pth", map_location=device, weights_only=False)
        if isinstance(state_dict, dict):
            newnet.load_state_dict(state_dict)
        else:
            # If it's a model object, extract state_dict
            newnet.load_state_dict(state_dict.state_dict())
    except:
        # If all else fails, load the entire model
        newnet = torch.load("best.pth", map_location=device, weights_only=False)
    
    newnet.to(device)
    newnet.eval()

    
    pred_test_fdssc = []
    with torch.no_grad():
        for X1, X2, y in all_iter:
            X1 = X1.to(device)
            X2 = X2.to(device)
            y_hat, _, _, _, _, _ = newnet(X1, X2)
            pred_test_fdssc.extend(np.array(y_hat.cpu().argmax(axis=1)))
    # toc2 = time.clock()

    Counter(pred_test_fdssc)
    gt_test = gt[total_indices]-1
    # gt_test = gt_test1[:-VAL_SIZE]
    # print('pre', np.unique(pred_test_fdssc))
    # print('gt_test', np.unique(gt_test[:-VAL_SIZE]))

    overall_acc_fdssc = metrics.accuracy_score(pred_test_fdssc, gt_test)
    confusion_matrix_fdssc = metrics.confusion_matrix(pred_test_fdssc, gt_test)
    # print(confusion_matrix_fdssc)
    each_acc_fdssc, average_acc_fdssc = aa_and_each_accuracy(confusion_matrix_fdssc)
    kappa = metrics.cohen_kappa_score(pred_test_fdssc, gt_test)

    # print("testing time:",toc2-tic2)
    savemat("river.mat", mdict={'result': pred_test_fdssc})
    KAPPA.append(kappa)
    OA.append(overall_acc_fdssc)
    AA.append(average_acc_fdssc)
    ELEMENT_ACC[index_iter, :] = each_acc_fdssc


    def calculate_metrics(confusion_matrix):
        # 从混淆矩阵中提取TP, TN, FP, FN
        TP = confusion_matrix[1, 1]
        TN = confusion_matrix[0, 0]
        FP = confusion_matrix[0, 1]
        FN = confusion_matrix[1, 0]

        # 计算Precision
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0

        # 计算Recall
        recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0

        # 计算F1 Score
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

        # 计算IoU (Intersection over Union)
        iou = TP / (TP + FP + FN) if (TP + FP + FN) > 0 else 0.0

        return precision, recall, f1, iou


    precision, recall, f1, iou = calculate_metrics(confusion_matrix_fdssc)

    print("-------- Training Finished-----------")
    print('OA:',OA)
    print('AA:',AA)
    print('Kappa:',KAPPA)
    print('OA_UN',each_acc_fdssc)
    print('precision',precision)
    print('recall',recall)
    print('f1',f1)
    print('iou',iou)


    generate_png(pred_test_fdssc, gt_hsi-1, 'Bay Area', total_indices)


